# radcoolpv — radiative cooling of silicon PV

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gsilvaoelker/radcoolpv-py/blob/main/radcoolpv.ipynb)

A module's emittance spectrum decides how hot it runs under the sun, and how hot
it runs decides how much power it makes. This notebook takes a spectrum — one you
measured, one you digitized from a paper, or one computed here from a geometry —
and returns the operating temperature, every term of the energy balance, and the
full set of PV parameters.

**Runtime → Run all** finishes on its own: about ten minutes to build the
optical solver, then a few minutes for the three cases. Then edit a YAML cell
and run it again. The model itself is written down in
[`docs/model.md`](https://github.com/gsilvaoelker/radcoolpv-py/blob/main/docs/model.md).

## 1. Set up the runtime

Colab runtimes are temporary: run this cell again after a reset. It installs
`radcoolpv` and builds the S4 optical solver from source (only Case B needs it).

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

PROJECT = Path("/content/radcoolpv-py")
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/gsilvaoelker/radcoolpv-py.git", str(PROJECT)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--editable", str(PROJECT)], check=True)
os.chdir(PROJECT)

# S4, the optical solver, from github.com/phoebe-p/S4 at the tested revision. About ten minutes.
if importlib.util.find_spec("S4") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y libboost-all-dev libfftw3-dev "
                   "liblapack-dev libopenblas-dev libsuitesparse-dev", shell=True, check=True)
    subprocess.run("git clone -q https://github.com/phoebe-p/S4.git /content/S4 && "
                   "git -C /content/S4 checkout -q 9569f5e555b967a4324eb1ea593d0f9f40761a61 && "
                   "make -C /content/S4 -j2 S4_pyext", shell=True, check=True)
    importlib.invalidate_caches()

from radcoolpv import config, pipeline, report
print("radcoolpv ready,", "with S4" if importlib.util.find_spec("S4") else "WITHOUT S4 (Case B will not run)")

## 2. Case A — a spectrum you already have → temperature and PV parameters

A case is a YAML file with up to three blocks. **A block is computed when it is
present**; delete `cell:` and you get the cooling curve alone.

| block | what it needs | why you would change it |
|---|---|---|
| `optics.file` | a text table, wavelength in **µm** in column 0 | your own spectrum |
| `optics.column` | which column is the hemispherical emittance | a measured or digitized file; **omit it** when the file is an `optics.txt` radcoolpv wrote, which already holds R, T, ε, A_Si and the atmospheric product |
| `thermal.ambient_temperature` | K | site conditions |
| `thermal.convection_coefficient` | W/m²·K, *everything* non-radiative: convection and conduction to the mount | the single most sensitive input; the paper states 6, a fit to its curves gives 12.5 |
| `cell.silicon_thickness` | µm | the spectrum does not know the stack, and the Auger term needs it |

**To use your own file:** click the folder icon on the left, upload it, and
change `file:` (and `column:`) below. A cell needs the spectrum to reach below
the band gap, 1.1 µm: a file that starts in the infrared has no sunlight to
convert, so drop the `cell:` block for it (Case A′).

The default here is the converged spectrum of the silica-microcylinder module of
Akerboom *et al.*, *ACS Photonics* **9**, 3831 (2022), so this cell reproduces
the paper's 327 K and 18.6 %.

In [ ]:
%%writefile case_a.yaml
optics:
  file: validation/data/C3_pv_cylinders.txt   # a radcoolpv optics.txt: no column needed

thermal:
  ambient_temperature: 300.0       # K
  convection_coefficient: 6.0      # W/m2-K, everything non-radiative

cell:
  silicon_thickness: 500.0         # um

In [ ]:
report.summary(pipeline.run(config.load("case_a.yaml")))

### Case A′ — a measured emittance → the cooling curve

The same block, pointed at a digitized measurement: the paper's Fig. 5a, one
column per surface. No `cell:`, because the file starts at 2 µm. The paper
states the absorbed sunlight directly, so `absorbed_solar_power` replaces the
AM1.5 integral, and `temperatures` sweeps the range the curve is wanted over.

In [ ]:
%%writefile case_measured.yaml
optics:
  file: validation/data/fig5a_measured_emittance.txt
  column: 3                        # 1 bare Au/Si, 2 flat silica, 3 cylinders

thermal:
  ambient_temperature: 300.0
  convection_coefficient: 12.54    # fitted to the paper's Fig. 5b; the paper states 6.0
  absorbed_solar_power: 808.0      # W/m2 absorbed, as the paper gives it
  temperatures: {min: 260.0, max: 380.0, n: 121}

In [ ]:
report.summary(pipeline.run(config.load("case_measured.yaml")))

## 3. Case B — a geometry → S4 → temperature and PV parameters

Replace `file:` with a structure and S4 computes the spectrum. The photonic
`geometry` sits on top of the `structure`, which sits on the semi-infinite
`substrate`; every material name used must appear in `materials`.

| key | meaning |
|---|---|
| `wavelength` | the grid every integral is taken on; it must stay inside every material table (the gold table ends at 24.93 µm) and resolve the 8–13 µm window |
| `angles` | `hemispherical` is required for the energy balance; `normal` for a quick optics-only look |
| `hemisphere_theta_points`, `hemisphere_azimuth_points` | the angular quadrature; one azimuth samples a single plane of incidence, which the validation uses; raise it when the response depends on azimuth |
| `s4_modes` | the Fourier truncation; a patterned layer needs 60–100+, a flat stack is exact at 1 |
| `geometry.shape` | `flat`, `cylinder {radius, height}`, `sphere` / `semisphere {radius, layers}`, `triangle {base, height, layers}`, `grating {duty, depth}` |
| `geometry.lattice` | `square {x}` or `hexagonal {x, y}` with x = y·√3 |

Cost scales as wavelengths × directions × 2 polarizations × modes³. The case
below is the paper's cylinder array on a **reduced grid** that runs in a few
minutes and lands at about 325 K and 18.5 %. The converged settings — `n: 1000`,
`hemisphere_theta_points: 8`, `s4_modes: 60`, in `validation/akerboom.yaml` —
take about an hour here and give 327.0 K and 18.64 %. That 2 K is what
convergence buys: a number worth reporting is one that stops moving when you
raise `s4_modes`, then `n`, then the angular grid.

In [ ]:
%%writefile case_b.yaml
optics:
  wavelength: {min: 0.3, max: 24.9, n: 300}     # converged: 1000
  angles: hemispherical
  hemisphere_theta_points: 4                     # converged: 8
  hemisphere_azimuth_points: 1
  s4_modes: 30                                   # converged: 60
  geometry:
    shape: cylinder
    photonic_material: sio2
    lattice: {type: hexagonal, x: 10.608811, y: 6.125}   # pitch 6.125 um
    cylinder: {radius: 1.75, height: 2.25}
  structure:
    - {material: sio2,    thickness: 500.0}
    - {material: silicon, thickness: 500.0}
    - {material: gold,    thickness: 0.08}
  substrate: vacuum
  materials:
    sio2: PalikKitamura_SiO2
    silicon: Palik_Si            # lossy; a lossless silicon makes no current
    gold: RII_Olmon_2012_ev_Au

thermal:
  ambient_temperature: 300.0
  convection_coefficient: 6.0

cell: {}                         # silicon_thickness is read from the structure

In [ ]:
report.summary(pipeline.run(config.load("case_b.yaml")))

## 4. Adding a material

Put a CSV with the header `lambda_um,n,k` into `radcoolpv/materials/data/`
(upload it through the folder icon) and name it, without `.csv`, in the
`materials:` block. The wavelength grid must stay inside its range; the error
names the file that is too narrow. `radcoolpv/materials/SOURCES.md` lists the
models that ship.

## 5. Reading the results

Every run writes `results/<case>_<time>/`:

- `optics.txt` — the spectrum: `lambda R T emit abs_si emit*emit_atm`. Feed it
  back to `optics.file` (no `column`) to re-run the thermal side without the
  solver. The sixth column is what makes that exact: the atmospheric term is
  the angular average of a *product*, which no averaged emittance can rebuild.
- `iv.csv`, `power.csv` (with a cell) or `cooling_power.csv` (without).
- `run.json` — the resolved case, input hashes, the Python and S4 used, and
  every scalar printed above. Quantities the run did not solve for are absent,
  not zero: a run without a cell reports no efficiency rather than an
  efficiency of zero. `silicon_from_emittance: true` marks a PV result whose
  silicon absorptance was inferred from a single emittance column rather than
  solved.
- `figures/`.

Sign convention: the balance solved is
P_rad(T) − P_atm + h(T − T_amb) − P_sun + P_MPP(T) + P_nt(T) = 0. Check it
against any paper before comparing numbers.